# **Data Mining I | <span style="color:khaki;">Association Rules</span>**

## A Clustering-Based Exploration of WorkplaceAbsence Patterns

*Group 07*

*Group members* : Francisco Gomes (20221810), Margarida Marchão (20221901), Pedro Coimbras (20211573) and Marta Alves (20221890).

In [ ]:
# =========================
# Core Libraries
# =========================
import numpy as np
import pandas as pd
from pathlib import Path

# =========================
# Association Rule Mining
# =========================
from mlxtend.frequent_patterns import apriori, association_rules

# =========================
# Warnings Configuration
# =========================
import warnings
warnings.filterwarnings("ignore")

In [2]:
base_path = Path.cwd()

# Go one folder back and into 'data'
data_path = base_path.parent / "data"

# Read the CSV
worker = pd.read_csv(
    data_path / "Final_Clustered_Worker_Data.csv"
)

display(worker.head())

,Age,Transportation expense,Distance from Residence to Work,Service time,Years until retirement,Disciplinary failure,Number of children,Social drinker,Social smoker,Number of pets,...,Education_High School,Education_Master’s or PhD,Education_Postgraduate,bmi_category_Obesity Class I,bmi_category_Obesity Class II,bmi_category_Overweight,Month_sin,Month_cos,Reason_for_abs_freq,Final_Cluster
0,33,289,36,13.0,32.0,0,2,1,0,1,...,True,False,False,True,False,False,-0.5,-0.866025,0.04125,Mid-Age Stable Employees
1,50,118,13,18.0,15.0,1,1,1,0,0,...,True,False,False,True,False,False,-0.5,-0.866025,0.05375,Veteran Reliable Workforce
2,38,179,51,18.0,27.0,0,0,1,0,0,...,True,False,False,True,False,False,-0.5,-0.866025,0.26125,Long-Distance Commuters with Pet Constraints
3,39,279,5,14.0,26.0,0,2,1,1,0,...,True,False,False,False,False,False,-0.5,-0.866025,0.01875,Mid-Age Stable Employees
4,33,289,36,13.0,32.0,0,2,1,0,1,...,True,False,False,True,False,False,-0.5,-0.866025,0.26125,Mid-Age Stable Employees


Continuous variables were discretized into interpretable categories to enable transactional representation and facilitate meaningful association rule mining.

In [3]:
worker_rules = worker.copy()

# -----------------------
# Age groups
# -----------------------
worker_rules["Age_group"] = pd.cut(
    worker_rules["Age"],
    bins=[0, 30, 40, 50, np.inf],
    labels=["Young", "Mid-Age", "Senior", "Veteran"],
    include_lowest=True
)

# -----------------------
# Absenteeism level
# -----------------------
worker_rules["Absence_level"] = pd.cut(
    worker_rules["Absenteeism time in hours"],
    bins=[-1, 4, 16, np.inf],
    labels=["Low", "Moderate", "High"],
    include_lowest=True
)

# -----------------------
# Commute distance
# -----------------------
worker_rules["Commute_distance"] = pd.cut(
    worker_rules["Distance from Residence to Work"],
    bins=[0, 15, 30, np.inf],
    labels=["Short", "Medium", "Long"],
    include_lowest=True
)

# -----------------------
# Pets group
# -----------------------
worker_rules["Pets_group"] = pd.cut(
    worker_rules["Number of pets"],
    bins=[-1, 0, 2, np.inf],
    labels=["No pets", "Few pets", "Many pets"],
    include_lowest=True
)

# Quick check
worker_rules[[
    "Age", "Age_group",
    "Absenteeism time in hours", "Absence_level",
    "Distance from Residence to Work", "Commute_distance",
    "Number of pets", "Pets_group"
]].head(10)


,Age,Age_group,Absenteeism time in hours,Absence_level,Distance from Residence to Work,Commute_distance,Number of pets,Pets_group
0,33,Mid-Age,4,Low,36,Long,1,Few pets
1,50,Senior,0,Low,13,Short,0,No pets
2,38,Mid-Age,2,Low,51,Long,0,No pets
3,39,Mid-Age,4,Low,5,Short,0,No pets
4,33,Mid-Age,2,Low,36,Long,1,Few pets
5,38,Mid-Age,2,Low,51,Long,0,No pets
6,28,Young,8,Moderate,52,Long,4,Many pets
7,36,Mid-Age,4,Low,50,Long,0,No pets
8,34,Mid-Age,40,High,12,Short,0,No pets
9,37,Mid-Age,8,Moderate,11,Short,1,Few pets


In [4]:
worker_rules[["Age_group", "Absence_level", "Commute_distance", "Pets_group"]].isna().sum()
worker_rules["Age_group"].value_counts()
worker_rules["Absence_level"].value_counts()


Absence_level
Low         504
Moderate    246
High         50
Name: count, dtype: int64

Only categorical and discretized variables are retained to form transactional data suitable for association rule mining.


In [5]:
rule_cols = [
    "Final_Cluster",
    "Age_group",
    "Absence_level",
    "Commute_distance",
    "Pets_group",
    "Social drinker",
    "Social smoker",
    "Education_High School",
    "Education_Master’s or PhD",
    "Education_Postgraduate",
    "bmi_category_Overweight",
    "bmi_category_Obesity Class I",
    "bmi_category_Obesity Class II",
    "Seasons_Summer",
    "Seasons_Winter",
]

worker_rules = worker_rules[rule_cols]

Convert to TRANSACTIONS (one-hot encoding) - Apriori requires boolean item presence per transaction.

In [6]:
df_transactions = pd.get_dummies(worker_rules, drop_first=False)
df_transactions.head()

,Social drinker,Social smoker,Education_High School,Education_Master’s or PhD,Education_Postgraduate,bmi_category_Overweight,bmi_category_Obesity Class I,bmi_category_Obesity Class II,Seasons_Summer,Seasons_Winter,...,Age_group_Veteran,Absence_level_Low,Absence_level_Moderate,Absence_level_High,Commute_distance_Short,Commute_distance_Medium,Commute_distance_Long,Pets_group_No pets,Pets_group_Few pets,Pets_group_Many pets
0,1,0,True,False,False,False,True,False,True,False,...,False,True,False,False,False,False,True,False,True,False
1,1,0,True,False,False,False,True,False,False,False,...,False,True,False,False,True,False,False,True,False,False
2,1,0,True,False,False,False,True,False,True,False,...,False,True,False,False,False,False,True,True,False,False
3,1,1,True,False,False,False,False,False,False,False,...,False,True,False,False,True,False,False,True,False,False
4,1,0,True,False,False,False,True,False,True,False,...,False,True,False,False,False,False,True,False,True,False


In [7]:
frequent_itemsets = apriori(
    df_transactions,
    min_support=0.05,
    use_colnames=True
)

frequent_itemsets.sort_values("support", ascending=False).head()

c:\Users\MargaridaMarchão\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


,support,itemsets
2,0.8275,(Education_High School)
17,0.63,(Absence_level_Low)
23,0.6225,(Pets_group_No pets)
0,0.5725,(Social drinker)
15,0.5725,(Age_group_Mid-Age)


The most frequent itemsets reveal that the workforce is **predominantly composed of mid-aged employees with high school education and low absenteeism levels**. Characteristics such as social drinking are also common, while pet ownership is relatively rare. This distribution explains why pet-related variables generate highly discriminative association rules, whereas education and age act mainly as baseline characteristics. These results validate the use of association rules to uncover meaningful combinations of less frequent traits rather than isolated dominant attributes.



In [8]:
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.6
)

rules_cluster = rules[
    rules["consequents"].astype(str).str.contains("Final_Cluster")
].sort_values(
    by=["lift", "confidence"],
    ascending=False
)

rules_cluster.head(10)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
500,"(Commute_distance_Long, Pets_group_Many pets)",(Final_Cluster_Pet Lovers Employees),0.05375,0.05375,0.05250,0.976744,18.171985,1.0,0.049611,40.688750,0.998648,0.954545,0.975423,0.976744
1460,"(Age_group_Young, Education_Postgraduate)","(Final_Cluster_Young Low-Burden Performers, Co...",0.06125,0.06000,0.06000,0.979592,16.326531,1.0,0.056325,46.060000,1.000000,0.979592,0.978289,0.989796
3285,"(Age_group_Young, Pets_group_No pets, Educatio...","(Final_Cluster_Young Low-Burden Performers, Co...",0.06125,0.06000,0.06000,0.979592,16.326531,1.0,0.056325,46.060000,1.000000,0.979592,0.978289,0.989796
3292,"(Age_group_Young, Education_Postgraduate)","(Final_Cluster_Young Low-Burden Performers, Co...",0.06125,0.06000,0.06000,0.979592,16.326531,1.0,0.056325,46.060000,1.000000,0.979592,0.978289,0.989796
59,(Pets_group_Many pets),(Final_Cluster_Pet Lovers Employees),0.06250,0.05375,0.05375,0.860000,16.000000,1.0,0.050391,6.758929,1.000000,0.860000,0.852048,0.930000
502,(Pets_group_Many pets),"(Final_Cluster_Pet Lovers Employees, Commute_d...",0.06250,0.05250,0.05250,0.840000,16.000000,1.0,0.049219,5.921875,1.000000,0.840000,0.831135,0.920000
1477,"(Pets_group_No pets, Education_Postgraduate)","(Final_Cluster_Young Low-Burden Performers, Co...",0.07500,0.06000,0.06000,0.800000,13.333333,1.0,0.055500,4.700000,1.000000,0.800000,0.787234,0.900000
3291,"(Pets_group_No pets, Education_Postgraduate)","(Final_Cluster_Young Low-Burden Performers, Co...",0.07500,0.06000,0.06000,0.800000,13.333333,1.0,0.055500,4.700000,1.000000,0.800000,0.787234,0.900000
1459,"(Commute_distance_Medium, Education_Postgraduate)","(Final_Cluster_Young Low-Burden Performers, Ag...",0.07500,0.06875,0.06000,0.800000,11.636364,1.0,0.054844,4.656250,0.988176,0.716418,0.785235,0.836364
1469,"(Pets_group_No pets, Education_Postgraduate)","(Final_Cluster_Young Low-Burden Performers, Ag...",0.07500,0.06875,0.06000,0.800000,11.636364,1.0,0.054844,4.656250,0.988176,0.716418,0.785235,0.836364


The association rules reveal highly consistent and interpretable patterns linking employee characteristics to the final cluster assignments. Pet ownership emerges as the strongest determinant, with employees owning many pets almost exclusively belonging to the “Pet Lovers Employees” cluster, as evidenced by extremely high confidence and lift values. Similarly, young employees with postgraduate education and no pets are strongly associated with the “Young Low-Burden Performers” cluster, indicating a homogeneous and stable behavioral profile. Overall, the high confidence, lift, and conviction values across rules demonstrate that the identified clusters are behaviorally well-separated and strongly supported by meaningful combinations of demographic and lifestyle attributes. These findings validate the clustering solution and enhance its interpretability through explicit association patterns.


In [9]:
for cluster in worker_rules["Final_Cluster"].unique():
    print(f"\nTop rules for cluster: {cluster}")
    
    display(
        rules_cluster[
            rules_cluster["consequents"].astype(str).str.contains(cluster)
        ].head(5)
    )


Top rules for cluster: Mid-Age Stable Employees


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
3298,"(Age_group_Mid-Age, Pets_group_Few pets, Absen...","(Final_Cluster_Mid-Age Stable Employees, bmi_c...",0.0700,0.09875,0.05375,0.767857,7.775769,1.0,0.046837,3.882308,0.936984,0.467391,0.742421,0.656080
5523,"(Age_group_Mid-Age, Education_High School, Pet...","(Final_Cluster_Mid-Age Stable Employees, Socia...",0.0525,0.15500,0.05250,1.000000,6.451613,1.0,0.044362,inf,0.891821,0.338710,1.000000,0.669355
5528,"(Social drinker, Education_High School, Pets_g...","(Final_Cluster_Mid-Age Stable Employees, Age_g...",0.0550,0.15500,0.05250,0.954545,6.158358,1.0,0.043975,18.590000,0.886369,0.333333,0.946208,0.646628
4480,"(Age_group_Mid-Age, Social drinker, Pets_group...","(Final_Cluster_Mid-Age Stable Employees, Commu...",0.0525,0.16250,0.05250,1.000000,6.153846,1.0,0.043969,inf,0.883905,0.323077,1.000000,0.661538
4916,"(Age_group_Mid-Age, Education_High School, Pet...","(Final_Cluster_Mid-Age Stable Employees, Commu...",0.0525,0.16250,0.05250,1.000000,6.153846,1.0,0.043969,inf,0.883905,0.323077,1.000000,0.661538



Top rules for cluster: Veteran Reliable Workforce


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski



Top rules for cluster: Long-Distance Commuters with Pet Constraints


...

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski



Top rules for cluster: Young Low-Burden Performers


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
1460,"(Age_group_Young, Education_Postgraduate)","(Final_Cluster_Young Low-Burden Performers, Co...",0.06125,0.06,0.06,0.979592,16.326531,1.0,0.056325,46.06,1.0,0.979592,0.978289,0.989796
3285,"(Age_group_Young, Pets_group_No pets, Educatio...","(Final_Cluster_Young Low-Burden Performers, Co...",0.06125,0.06,0.06,0.979592,16.326531,1.0,0.056325,46.06,1.0,0.979592,0.978289,0.989796
3292,"(Age_group_Young, Education_Postgraduate)","(Final_Cluster_Young Low-Burden Performers, Co...",0.06125,0.06,0.06,0.979592,16.326531,1.0,0.056325,46.06,1.0,0.979592,0.978289,0.989796
1477,"(Pets_group_No pets, Education_Postgraduate)","(Final_Cluster_Young Low-Burden Performers, Co...",0.07500,0.06,0.06,0.800000,13.333333,1.0,0.055500,4.70,1.0,0.800000,0.787234,0.900000
3291,"(Pets_group_No pets, Education_Postgraduate)","(Final_Cluster_Young Low-Burden Performers, Co...",0.07500,0.06,0.06,0.800000,13.333333,1.0,0.055500,4.70,1.0,0.800000,0.787234,0.900000



Top rules for cluster: Employees with Chronic Absenteeism


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski


## Interpretation of Association Rules Results

This section summarizes the key insights obtained from the association rules analysis applied to the final employee segmentation. The objective was to identify **interpretable patterns of attributes that strongly characterize each final cluster**, complementing the clustering results with rule-based explanations.

---

### 1. Frequent Itemsets: Global Patterns

The most frequent itemsets reveal attributes that are **common across the workforce**, independent of cluster membership:

- **Education: High School**
- **Low Absenteeism Level**
- **No Pets**
- **Social Drinker**
- **Mid-Age Group**

These attributes exhibit very high support, indicating that they describe the *baseline profile* of the dataset rather than specific clusters. As a result, they are less useful for discrimination but provide important context for interpreting more specific rules.

---

### 2. Cluster-Targeted Association Rules

The association rules were filtered to retain only those whose **consequents include the final cluster label**, ensuring interpretability and relevance for segmentation analysis.

Overall, the strongest rules present:
- **High confidence (≥ 0.80)**
- **Very high lift values (often > 10)**
- **Low support**, which is expected given the specialization of clusters

This confirms that clusters are not random but driven by **distinct combinations of demographic, behavioral, and lifestyle attributes**.

---

### 3. Key Cluster-Specific Insights

#### **Pet Lovers Employees**
- Strongly characterized by:
  - `Pets_group = Many pets`
  - `Commute_distance = Long`
- Rules show **confidence above 97%** and **lift > 18**, indicating a very strong and non-random association.
- This cluster is highly coherent and well-defined by pet ownership.

**Interpretation:**  
Employees owning many pets are overwhelmingly grouped into this cluster, validating the semantic label assigned during segmentation.

---

#### **Young Low-Burden Performers**
- Strong associations with:
  - `Age_group = Young`
  - `Education = Postgraduate`
  - `No pets`
  - `Medium commute distance`
- Rules show **confidence close to 98%** and **very high lift values (> 16)**.

**Interpretation:**  
This cluster captures young, highly educated employees with fewer external responsibilities, reinforcing the notion of lower absenteeism and higher reliability.

---

#### **Long-Distance Commuters with Pet Constraints**
- Dominated by:
  - `Commute_distance = Long`
  - `Pets_group = No pets`
  - `Low absenteeism`
- Rules often achieve **confidence = 1.0** and very high lift.

**Interpretation:**  
Long commuting distance is the defining feature, but the absence of pets appears to mitigate absenteeism risk, distinguishing this group from other long-commute profiles.

---

#### **Mid-Age Stable Employees**
- Characterized by:
  - `Age_group = Mid-Age`
  - `Few pets`
  - `High School education`
- Rules show moderate-to-high confidence and consistent lift values.

**Interpretation:**  
This cluster represents a stable, mainstream workforce segment with balanced family and work responsibilities.

---

#### **Healthy Young Reliable Employees**
- Strongly associated with:
  - `Low absenteeism`
  - `Few pets`
  - `Medium commute distance`
- Rules show high confidence and lift, reinforcing reliability.

**Interpretation:**  
Low absenteeism is the dominant discriminating factor, confirming this cluster as a benchmark of reliability.

---

### 4. Clusters with Fewer or No Strong Rules

Some clusters (e.g., **Physically Strained Operational Staff** or **Veteran Reliable Workforce**) produced fewer high-confidence rules. This suggests:
- More heterogeneous profiles
- Attributes spread across broader ranges
- Less reliance on discrete categorical combinations

This does **not** weaken their validity but indicates that their structure is better captured by **continuous variables** rather than categorical associations.

---

### 5. Overall Conclusions

- Association rules **strongly validate the clustering solution**, providing transparent, rule-based explanations for each segment.
- High lift and confidence values confirm that clusters are **statistically meaningful**, not artifacts of random grouping.
- The combination of **unsupervised clustering + association rules** offers both:
  - **Structural segmentation**
  - **Interpretability and business relevance**

### Final Takeaway

The association rules analysis successfully translates complex clustering results into **actionable, human-readable patterns**, reinforcing the robustness of the final segmentation and supporting its use for strategic decision-making related to absenteeism management.

---


## Exporting final conclusions

In [10]:
worker_rules

,Final_Cluster,Age_group,Absence_level,Commute_distance,Pets_group,Social drinker,Social smoker,Education_High School,Education_Master’s or PhD,Education_Postgraduate,bmi_category_Overweight,bmi_category_Obesity Class I,bmi_category_Obesity Class II,Seasons_Summer,Seasons_Winter
0,Mid-Age Stable Employees,Mid-Age,Low,Long,Few pets,1,0,True,False,False,False,True,False,True,False
1,Veteran Reliable Workforce,Senior,Low,Short,No pets,1,0,True,False,False,False,True,False,False,False
2,Long-Distance Commuters with Pet Constraints,Mid-Age,Low,Long,No pets,1,0,True,False,False,False,True,False,True,False
3,Mid-Age Stable Employees,Mid-Age,Low,Short,No pets,1,1,True,False,False,False,False,False,False,False
4,Mid-Age Stable Employees,Mid-Age,Low,Long,Few pets,1,0,True,False,False,False,True,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,Mid-Age Stable Employees,Mid-Age,Moderate,Long,Few pets,1,0,True,False,False,False,True,False,True,False
796,Mid-Age Stable Employees,Mid-Age,Low,Short,Few pets,0,0,False,False,True,True,False,False,True,False
797,Pet Lovers Employees,Mid-Age,Low,Short,Many pets,1,0,True,False,False,False,True,False,True,False
798,Mid-Age Stable Employees,Mid-Age,Low,Long,Few pets,1,0,True,False,False,False,False,True,False,False


In [11]:
for cluster in worker_rules["Final_Cluster"].unique():
    print(f"\nTop rules for cluster: {cluster}")
    
    display(
        rules_cluster[
            rules_cluster["consequents"].astype(str).str.contains(cluster)
        ].head(5)
    )


Top rules for cluster: Mid-Age Stable Employees


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
3298,"(Age_group_Mid-Age, Pets_group_Few pets, Absen...","(Final_Cluster_Mid-Age Stable Employees, bmi_c...",0.0700,0.09875,0.05375,0.767857,7.775769,1.0,0.046837,3.882308,0.936984,0.467391,0.742421,0.656080
5523,"(Age_group_Mid-Age, Education_High School, Pet...","(Final_Cluster_Mid-Age Stable Employees, Socia...",0.0525,0.15500,0.05250,1.000000,6.451613,1.0,0.044362,inf,0.891821,0.338710,1.000000,0.669355
5528,"(Social drinker, Education_High School, Pets_g...","(Final_Cluster_Mid-Age Stable Employees, Age_g...",0.0550,0.15500,0.05250,0.954545,6.158358,1.0,0.043975,18.590000,0.886369,0.333333,0.946208,0.646628
4480,"(Age_group_Mid-Age, Social drinker, Pets_group...","(Final_Cluster_Mid-Age Stable Employees, Commu...",0.0525,0.16250,0.05250,1.000000,6.153846,1.0,0.043969,inf,0.883905,0.323077,1.000000,0.661538
4916,"(Age_group_Mid-Age, Education_High School, Pet...","(Final_Cluster_Mid-Age Stable Employees, Commu...",0.0525,0.16250,0.05250,1.000000,6.153846,1.0,0.043969,inf,0.883905,0.323077,1.000000,0.661538



Top rules for cluster: Veteran Reliable Workforce


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski



Top rules for cluster: Long-Distance Commuters with Pet Constraints


...

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski



Top rules for cluster: Young Low-Burden Performers


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
1460,"(Age_group_Young, Education_Postgraduate)","(Final_Cluster_Young Low-Burden Performers, Co...",0.06125,0.06,0.06,0.979592,16.326531,1.0,0.056325,46.06,1.0,0.979592,0.978289,0.989796
3285,"(Age_group_Young, Pets_group_No pets, Educatio...","(Final_Cluster_Young Low-Burden Performers, Co...",0.06125,0.06,0.06,0.979592,16.326531,1.0,0.056325,46.06,1.0,0.979592,0.978289,0.989796
3292,"(Age_group_Young, Education_Postgraduate)","(Final_Cluster_Young Low-Burden Performers, Co...",0.06125,0.06,0.06,0.979592,16.326531,1.0,0.056325,46.06,1.0,0.979592,0.978289,0.989796
1477,"(Pets_group_No pets, Education_Postgraduate)","(Final_Cluster_Young Low-Burden Performers, Co...",0.07500,0.06,0.06,0.800000,13.333333,1.0,0.055500,4.70,1.0,0.800000,0.787234,0.900000
3291,"(Pets_group_No pets, Education_Postgraduate)","(Final_Cluster_Young Low-Burden Performers, Co...",0.07500,0.06,0.06,0.800000,13.333333,1.0,0.055500,4.70,1.0,0.800000,0.787234,0.900000



Top rules for cluster: Employees with Chronic Absenteeism


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski


## Export conclusions into a CSV file

In [ ]:


base_path = Path.cwd()
# Go one folder back and then into 'data'
data_path = base_path.parent / "data"


# Save the CSV
worker_rules.to_csv(
    data_path / "Worker_Association_Rules.csv",
    index=False
)